# Dataset EDA: Specimen Distribution and Leakage Risk

This notebook summarizes the three dataset sources used in the paper plan:

- Mendeley classification dataset: https://data.mendeley.com/datasets/jhrtdj9txm/3
- EWU Roboflow segmentation dataset: https://universe.roboflow.com/shirmpdiseasedtection/ewu_shrimp_disease
- Hand-labeled Roboflow segmentation dataset: https://universe.roboflow.com/lets-try-this/shrimpdishandsegv2

The goal is not to train a model. The goal is to produce paper-ready evidence about dataset distribution, specimen grouping, label/mask counts, and leakage risk.


In [ ]:
from pathlib import Path
import os
import re
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'])
    import yaml

try:
    import matplotlib.pyplot as plt
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'])
    import matplotlib.pyplot as plt

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
REPORT_DIR = Path('/kaggle/working/shrimp_paper_eda_reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Reports will be saved to: {REPORT_DIR}')


## Configure Dataset Paths

Fill these paths after downloading or attaching datasets in Kaggle. Leave a path empty if that dataset is not available in the current session.

For Roboflow datasets, you may either download them in a previous cell/notebook or use the optional download cell below.


In [ ]:
# Update these paths in Kaggle/local runtime.
MENDELEY_ROOT = Path('')  # example: Path('/kaggle/input/shrimp-disease-classification')
EWU_ROOT = Path('')       # example: Path('/kaggle/working/EWU_Shrimp_Disease-1')
HAND_ROOT = Path('/kaggle/working/shrimpDisHandSegV2-1')

DATASET_CONFIGS = [
    {
        'dataset_key': 'mendeley_classification',
        'display_name': 'Mendeley classification dataset',
        'task': 'classification',
        'root': MENDELEY_ROOT,
        'parser': 'mendeley_hand',
        'source_url': 'https://data.mendeley.com/datasets/jhrtdj9txm/3',
    },
    {
        'dataset_key': 'ewu_segmentation',
        'display_name': 'EWU Roboflow segmentation dataset',
        'task': 'segmentation',
        'root': EWU_ROOT,
        'parser': 'ewu',
        'source_url': 'https://universe.roboflow.com/shirmpdiseasedtection/ewu_shrimp_disease',
    },
    {
        'dataset_key': 'hand_labeled_segmentation',
        'display_name': 'Hand-labeled Roboflow segmentation dataset',
        'task': 'segmentation',
        'root': HAND_ROOT,
        'parser': 'mendeley_hand',
        'source_url': 'https://universe.roboflow.com/lets-try-this/shrimpdishandsegv2',
    },
]

for cfg in DATASET_CONFIGS:
    print(cfg['dataset_key'], '->', cfg['root'], 'exists=', bool(str(cfg['root'])) and cfg['root'].exists())


## Optional Roboflow Download

Use this only if the datasets are not already available. Keep API keys in Kaggle Secrets when possible.


In [ ]:
# Optional: uncomment and configure to download Roboflow datasets.
# import importlib.util, subprocess, sys, os
# if importlib.util.find_spec('roboflow') is None:
#     subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'])
# from roboflow import Roboflow
#
# def get_roboflow_api_key():
#     try:
#         from kaggle_secrets import UserSecretsClient
#         key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
#         if key:
#             return key
#     except Exception:
#         pass
#     return os.environ.get('ROBOFLOW_API_KEY', '').strip()
#
# rf = Roboflow(api_key=get_roboflow_api_key())
#
# # Hand-labeled dataset
# hand_project = rf.workspace('lets-try-this').project('shrimpdishandsegv2')
# hand_dataset = hand_project.version(1).download('yolo26')
# HAND_ROOT = Path(hand_dataset.location)
#
# # EWU dataset: verify workspace/project/version in Roboflow UI before running.
# ewu_project = rf.workspace('shirmpdiseasedtection').project('ewu_shrimp_disease')
# ewu_dataset = ewu_project.version(1).download('yolo26')
# EWU_ROOT = Path(ewu_dataset.location)


## Filename Parsers

Mendeley and the hand-labeled dataset use:

```text
<ShrimpDisease>-<ShrimpID>-img-<imgnum>.jpg
```

EWU uses the old leakage-fixed training parser:

```text
Shrimp_<shrimpid>-<imgnum>.jpg
Shrimp_<shrimpid>-<imgnum>-.jpg
Shrimp_<shrimpid>.jpg
```

Roboflow may export these as names such as `Shrimp_318-1-_jpg.rf.<hash>.jpg`. The specimen key ignores image number so all photos from one shrimp stay together. The parser also accepts the earlier observed parenthesized form `Shrimp_<shrimpid> (<imgnum>).jpg` as a fallback.


In [ ]:
MENDELEY_HAND_PATTERN = re.compile(
    r'^(?P<disease>.+)-(?P<shrimp_id>[^-]+)-img-(?P<img_num>\d+)$',
    re.IGNORECASE,
)

EWU_WITH_IMAGE_NUM_PATTERN = re.compile(
    r'^Shrimp_(?P<shrimp_id>.+?)-(?P<img_num>\d+)-?$',
    re.IGNORECASE,
)
EWU_SINGLE_IMAGE_PATTERN = re.compile(
    r'^Shrimp_(?P<shrimp_id>.+)$',
    re.IGNORECASE,
)
EWU_PAREN_PATTERN = re.compile(
    r'^Shrimp_(?P<shrimp_id>[^\s()]+)\s*\((?P<img_num>\d+)\)$',
    re.IGNORECASE,
)


def normalize_roboflow_stem(stem):
    stem = re.sub(r'_(jpg|jpeg|png|bmp|webp)\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    stem = re.sub(r'\.rf\.[0-9a-f]+$', '', stem, flags=re.IGNORECASE)
    return stem


def parse_image_name(path, parser):
    stem = normalize_roboflow_stem(Path(path).stem)
    if parser == 'mendeley_hand':
        match = MENDELEY_HAND_PATTERN.match(stem)
        if match:
            disease = match.group('disease')
            shrimp_id = match.group('shrimp_id')
            img_num = int(match.group('img_num'))
            return {
                'parse_ok': True,
                'disease_from_name': disease,
                'shrimp_id': shrimp_id,
                'img_num': img_num,
                'specimen_key': f'{disease.lower()}::{shrimp_id}',
                'normalized_stem': stem,
            }
    elif parser == 'ewu':
        match = EWU_WITH_IMAGE_NUM_PATTERN.match(stem)
        if match:
            shrimp_id = match.group('shrimp_id')
            img_num = int(match.group('img_num'))
        else:
            match = EWU_PAREN_PATTERN.match(stem)
            if match:
                shrimp_id = match.group('shrimp_id')
                img_num = int(match.group('img_num'))
            else:
                match = EWU_SINGLE_IMAGE_PATTERN.match(stem)
                if not match:
                    shrimp_id = None
                    img_num = None
                else:
                    shrimp_id = match.group('shrimp_id')
                    img_num = 1
        if shrimp_id is not None:
            return {
                'parse_ok': True,
                'disease_from_name': 'Shrimp',
                'shrimp_id': shrimp_id,
                'img_num': img_num,
                'specimen_key': f'shrimp::{shrimp_id}',
                'normalized_stem': stem,
            }
    return {
        'parse_ok': False,
        'disease_from_name': None,
        'shrimp_id': None,
        'img_num': None,
        'specimen_key': f'unparsed::{stem}',
        'normalized_stem': stem,
    }


def find_images(root):
    root = Path(root)
    if not root.exists():
        return []
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def split_from_path(path):
    parts = [p.lower() for p in Path(path).parts]
    for split in ['train', 'valid', 'val', 'test']:
        if split in parts:
            return 'valid' if split == 'val' else split
    return 'all'


## Segmentation Label Utilities

For YOLO segmentation, empty label files are treated as healthy negative images. Non-empty label files indicate disease mask instances.


In [ ]:
def load_class_names(dataset_root):
    dataset_root = Path(dataset_root)
    yaml_candidates = list(dataset_root.glob('data.yaml')) + list(dataset_root.rglob('data.yaml'))
    if not yaml_candidates:
        return []
    with open(yaml_candidates[0], 'r') as f:
        data = yaml.safe_load(f)
    names = data.get('names', [])
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    return list(names)


def label_path_for_image(image_path):
    image_path = Path(image_path)
    parts = list(image_path.parts)
    if 'images' in parts:
        idx = parts.index('images')
        label_parts = parts[:idx] + ['labels'] + parts[idx + 1:]
        return Path(*label_parts).with_suffix('.txt')
    return image_path.with_suffix('.txt')


def read_yolo_label_stats(image_path, class_names):
    label_path = label_path_for_image(image_path)
    if not label_path.exists():
        return {
            'label_exists': False,
            'is_labeled_disease': False,
            'mask_instances': 0,
            'class_ids': [],
            'class_names': [],
        }
    lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    class_ids = []
    for line in lines:
        try:
            class_ids.append(int(float(line.split()[0])))
        except Exception:
            pass
    names = [class_names[c] if c < len(class_names) else f'class_{c}' for c in class_ids]
    return {
        'label_exists': True,
        'is_labeled_disease': bool(lines),
        'mask_instances': len(lines),
        'class_ids': class_ids,
        'class_names': names,
    }


## Build Image-Level EDA Table

In [ ]:
all_rows = []

for cfg in DATASET_CONFIGS:
    root = Path(cfg['root'])
    if not str(root) or not root.exists():
        print(f"Skipping {cfg['dataset_key']}: root not found -> {root}")
        continue

    class_names = load_class_names(root) if cfg['task'] == 'segmentation' else []
    images = find_images(root)
    print(f"{cfg['dataset_key']}: found {len(images)} images")

    for image_path in images:
        parsed = parse_image_name(image_path, cfg['parser'])
        row = {
            'dataset_key': cfg['dataset_key'],
            'display_name': cfg['display_name'],
            'task': cfg['task'],
            'source_url': cfg['source_url'],
            'root': str(root),
            'split': split_from_path(image_path),
            'image_path': str(image_path),
            'image_name': image_path.name,
            **parsed,
        }

        if cfg['task'] == 'segmentation':
            seg = read_yolo_label_stats(image_path, class_names)
            row.update({
                'label_exists': seg['label_exists'],
                'is_labeled_disease': seg['is_labeled_disease'],
                'mask_instances': seg['mask_instances'],
                'mask_class_ids': ','.join(map(str, seg['class_ids'])),
                'mask_class_names': ','.join(seg['class_names']),
            })
        else:
            row.update({
                'label_exists': None,
                'is_labeled_disease': None,
                'mask_instances': 0,
                'mask_class_ids': '',
                'mask_class_names': '',
            })

        all_rows.append(row)

image_df = pd.DataFrame(all_rows)
image_csv = REPORT_DIR / 'dataset_image_level_eda.csv'
image_df.to_csv(image_csv, index=False)
print(f'Saved image-level EDA: {image_csv}')
display(image_df.head())
print(image_df.shape)


## Dataset-Level Summary

In [ ]:
def safe_nunique(series):
    return int(series.dropna().nunique()) if len(series) else 0

summary_rows = []
if not image_df.empty:
    for dataset_key, g in image_df.groupby('dataset_key'):
        seg = g[g['task'] == 'segmentation']
        summary_rows.append({
            'dataset_key': dataset_key,
            'display_name': g['display_name'].iloc[0],
            'task': g['task'].iloc[0],
            'source_url': g['source_url'].iloc[0],
            'images': len(g),
            'parsed_images': int(g['parse_ok'].sum()),
            'unparsed_images': int((~g['parse_ok']).sum()),
            'specimens': safe_nunique(g['specimen_key']),
            'shrimp_ids': safe_nunique(g['shrimp_id']),
            'diseases_from_filename': safe_nunique(g['disease_from_name']),
            'seg_labeled_disease_images': int(seg['is_labeled_disease'].fillna(False).sum()) if not seg.empty else None,
            'seg_healthy_or_empty_label_images': int((seg['label_exists'].fillna(False) & ~seg['is_labeled_disease'].fillna(False)).sum()) if not seg.empty else None,
            'seg_missing_label_images': int((~seg['label_exists'].fillna(False)).sum()) if not seg.empty else None,
            'seg_mask_instances': int(seg['mask_instances'].sum()) if not seg.empty else None,
            'mean_images_per_specimen': float(g.groupby('specimen_key').size().mean()) if len(g) else 0,
            'max_images_per_specimen': int(g.groupby('specimen_key').size().max()) if len(g) else 0,
        })

summary_df = pd.DataFrame(summary_rows)
summary_csv = REPORT_DIR / 'dataset_distribution_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Saved dataset summary: {summary_csv}')
display(summary_df)


## Split Distribution and Leakage Risk

This section checks whether inferred specimens occur in multiple splits. For a fair evaluation, each `specimen_key` should appear in only one split.


In [ ]:
split_rows = []
leakage_rows = []

if not image_df.empty:
    for dataset_key, g in image_df.groupby('dataset_key'):
        for split, sg in g.groupby('split'):
            split_rows.append({
                'dataset_key': dataset_key,
                'split': split,
                'images': len(sg),
                'specimens': sg['specimen_key'].nunique(),
                'labeled_disease_images': int(sg['is_labeled_disease'].fillna(False).sum()) if sg['task'].iloc[0] == 'segmentation' else None,
                'mask_instances': int(sg['mask_instances'].sum()) if sg['task'].iloc[0] == 'segmentation' else None,
            })

        for specimen_key, gg in g.groupby('specimen_key'):
            splits = sorted(set(gg['split']))
            if len(splits) > 1 and set(splits) != {'all'}:
                leakage_rows.append({
                    'dataset_key': dataset_key,
                    'specimen_key': specimen_key,
                    'splits': ','.join(splits),
                    'images': len(gg),
                    'example_images': '; '.join(gg['image_name'].head(5).tolist()),
                })

split_df = pd.DataFrame(split_rows)
leakage_df = pd.DataFrame(leakage_rows)

split_csv = REPORT_DIR / 'dataset_split_distribution.csv'
leakage_csv = REPORT_DIR / 'dataset_split_leakage_candidates.csv'
split_df.to_csv(split_csv, index=False)
leakage_df.to_csv(leakage_csv, index=False)
print(f'Saved split distribution: {split_csv}')
print(f'Saved leakage candidates: {leakage_csv}')
display(split_df)
display(leakage_df.head(20))
print('Potential leaking specimen groups:', len(leakage_df))


## Class and Mask Distribution

In [ ]:
class_rows = []
if not image_df.empty:
    for _, row in image_df[image_df['task'] == 'segmentation'].iterrows():
        names = [n for n in str(row.get('mask_class_names', '')).split(',') if n]
        if not names and row.get('label_exists') and not row.get('is_labeled_disease'):
            class_rows.append({
                'dataset_key': row['dataset_key'],
                'split': row['split'],
                'class_name': 'healthy_empty_label',
                'instances': 0,
                'images': 1,
            })
        for name in names:
            class_rows.append({
                'dataset_key': row['dataset_key'],
                'split': row['split'],
                'class_name': name,
                'instances': 1,
                'images': 1,
            })

class_df_raw = pd.DataFrame(class_rows)
if not class_df_raw.empty:
    class_df = class_df_raw.groupby(['dataset_key', 'split', 'class_name'], as_index=False).agg(
        instances=('instances', 'sum'),
        image_mentions=('images', 'sum'),
    )
else:
    class_df = pd.DataFrame(columns=['dataset_key', 'split', 'class_name', 'instances', 'image_mentions'])

class_csv = REPORT_DIR / 'dataset_class_mask_distribution.csv'
class_df.to_csv(class_csv, index=False)
print(f'Saved class/mask distribution: {class_csv}')
display(class_df)


## Specimen Size Distribution

In [ ]:
specimen_rows = []
if not image_df.empty:
    for (dataset_key, specimen_key), g in image_df.groupby(['dataset_key', 'specimen_key']):
        specimen_rows.append({
            'dataset_key': dataset_key,
            'specimen_key': specimen_key,
            'shrimp_id': g['shrimp_id'].iloc[0],
            'disease_from_name': g['disease_from_name'].iloc[0],
            'images': len(g),
            'splits': ','.join(sorted(set(g['split']))),
            'mask_instances': int(g['mask_instances'].sum()),
            'labeled_disease_images': int(g['is_labeled_disease'].fillna(False).sum()),
            'example_images': '; '.join(g['image_name'].head(5).tolist()),
        })

specimen_df = pd.DataFrame(specimen_rows).sort_values(['dataset_key', 'images'], ascending=[True, False]) if specimen_rows else pd.DataFrame()
specimen_csv = REPORT_DIR / 'dataset_specimen_distribution.csv'
specimen_df.to_csv(specimen_csv, index=False)
print(f'Saved specimen distribution: {specimen_csv}')
display(specimen_df.head(30))


## Plots

In [ ]:
if not summary_df.empty:
    ax = summary_df.set_index('dataset_key')[['images', 'specimens']].plot(kind='bar', figsize=(10, 5))
    ax.set_title('Images and inferred specimens by dataset')
    ax.set_ylabel('count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plot_path = REPORT_DIR / 'dataset_images_specimens.png'
    plt.savefig(plot_path, dpi=160)
    plt.show()
    print(f'Saved plot: {plot_path}')

if not specimen_df.empty:
    for dataset_key, g in specimen_df.groupby('dataset_key'):
        plt.figure(figsize=(8, 4))
        plt.hist(g['images'], bins=range(1, int(g['images'].max()) + 2), align='left')
        plt.title(f'Images per inferred specimen: {dataset_key}')
        plt.xlabel('images per specimen')
        plt.ylabel('specimen count')
        plt.tight_layout()
        plot_path = REPORT_DIR / f'{dataset_key}_images_per_specimen_hist.png'
        plt.savefig(plot_path, dpi=160)
        plt.show()
        print(f'Saved plot: {plot_path}')


## Paper-Ready Notes

Use these outputs to justify:

- why specimen grouping is necessary
- whether EWU is reliable enough for optimization claims
- whether hand-labeled data has different distribution from Mendeley/EWU
- how many specimens/images/masks support the experiments


In [ ]:
print('Generated report files:')
for p in sorted(REPORT_DIR.glob('*')):
    print('-', p)
